In [1]:
import sys
from IPython.display import display, Javascript

def restart_kernel():
    """Restart the Jupyter Notebook kernel to reflect changes in modules and packages."""
    display(Javascript("Jupyter.notebook.kernel.restart()"))
    print("Kernel is restarting...")

restart_kernel()

import numpy as np
import torch
import tensorly as tl
from tensorly.decomposition import parafac
import sparse
import pickle
import os
import numpy.random as rn
import scipy.stats as st
from tensorly.cp_tensor import CPTensor

from bptf import BPTF as BPTF
import bptf

# Please run everything up to the first bptf fit
# Thereafter, run each following cell on their own

<IPython.core.display.Javascript object>

Kernel is restarting...


# Helper functions

In [2]:
def generate(shp=(30, 30, 20, 10), K=5, alpha=0.1, beta=0.1):
    """Generate a count tensor from the BPTF model.

    PARAMS:
    shp -- (tuple) shape of the generated count tensor
    K -- (int) number of latent components
    alpha -- (float) shape parameter of gamma prior over factors
    beta -- (float) rate parameter of gamma prior over factors

    RETURNS:
    Mu -- (np.ndarray) true Poisson rates
    Y -- (np.ndarray) generated count tensor
    """
    Theta_DK_M = [rn.gamma(alpha, 1./beta, size=(D, K)) for D in shp]
    Mu = tl.cp_to_tensor(CPTensor((None, Theta_DK_M)))
    assert Mu.shape == shp
    Y = rn.poisson(Mu)
    return Mu, Y

# Load data

In [3]:
use_existing_data = False

# for the first bptf.fit:
# using seed 100 causes the assert delta >= 0 to trip
# but using seed 0 doesn't
# please swap between the 2 seeds to get the 2 different types of assertion errors in the other cells
rn.seed(0)

if use_existing_data:
    assert os.path.exists('sptensor.pkl'), 'No such file.'
    with open('sptensor.pkl', 'rb') as f:
        data = pickle.load(f)
    data = data[:, :, :, :12, :]
    data = sparse.COO(data)
else:
    data = generate(shp=(200, 200, 20, 12, 3), K=10)[1]
    data = sparse.COO(data)

# Building mask (base example)

In [4]:
mask = np.zeros(data.shape)
# april of GDELT is set to missing
mask[:, :, :, 3, 1] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

# Fit model

In [5]:
n_components = 10
max_iter = 100
tol = 1e-10

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=1, tol=tol)

  5%|▌         | 5/100 [00:31<09:54,  6.26s/it]


KeyboardInterrupt: 

# Mask with last mode completely masked

In [ ]:
mask = np.zeros(data.shape)
# april of GDELT is set to missing
mask[:, :, :, 3, :] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=1, tol=tol)

  5%|▌         | 5/100 [00:41<13:08,  8.30s/it]C:\Users\luiyu\OneDrive - The University of Chicago\UChicago_MS_Stat\aaron_schein\bptf_new\bptf\src\bptf\bptf.py:200: RuntimeWarning: invalid value encountered in log
  self.G_DK_M[m] = np.exp(sp.psi(shp_DK) - np.log(rte_DK))
  5%|▌         | 5/100 [00:47<14:56,  9.44s/it]


AssertionError: 

# Mask with the 1st and 3rd indices masked

In [ ]:
mask = np.zeros(data.shape)
# april is set to missing
mask[:, :, :, 3, [0, 2]] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=1, tol=tol)

 16%|█▌        | 16/100 [01:37<08:28,  6.05s/it]C:\Users\luiyu\OneDrive - The University of Chicago\UChicago_MS_Stat\aaron_schein\bptf_new\bptf\src\bptf\bptf.py:200: RuntimeWarning: invalid value encountered in log
  self.G_DK_M[m] = np.exp(sp.psi(shp_DK) - np.log(rte_DK))
 16%|█▌        | 16/100 [01:41<08:51,  6.33s/it]


AssertionError: 

# Mask with only diagonals masked

In [ ]:
mask = np.zeros(data.shape)

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=1, tol=tol)

100%|██████████| 100/100 [05:27<00:00,  3.27s/it]


BPTF(data_shape=(200, 200, 20, 12, 3), n_components=10)